# Phase 5 — Mangrove Reforestation Mission

Adapts the drone fleet system for **seed-planting** missions inspired by Distant Imagery's
coastal mangrove reforestation work near Abu Dhabi.

**What's new vs. the spraying system:**
- `MissionConfig` — single user-facing config dataclass (seed capacity, spacing, battery life, scale)
- Seed capacity lifecycle — drones deplete seed hopper and return to dock to refill, same as battery
- Soil-only planting — `detect_soil_mask()` excludes water channels and existing canopy
- Non-straight-line planting — Gaussian jitter on recorded seed positions
- Reforestation metrics — seeds planted, area covered (m²), expected survivors
- Tidal event support — `apply_tidal_mask()` temporarily excludes flooded cells

**Everything else is unchanged:** MILP optimizer, three-tier planner, failure injection, replanning, animation.

---
**Sections**
1. Configuration
2. Synthetic mangrove field
3. Soil mask (synthetic demo + real image walkthrough)
4. Strip generation
5. Fleet assignment (MILP)
6. Simulation with seed tracking
7. Reforestation metrics
8. Animation
9. Seed drop scatter
10. Tidal event — dynamic mask + replanning
11. Failure + replanning
12. Monte Carlo worst-case
13. Summary

In [ ]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# Core modules (unchanged)
from src.field.generator import generate_strips, synthetic_field
from src.field.ingest    import load_image_grid, load_image_as_array
from src.optimizer.milp  import DroneSpec, assign_strips
from src.optimizer.planner import plan, PlannerContext
from src.simulation.engine  import simulate
from src.simulation.metrics import (
    monte_carlo_analysis, plot_monte_carlo,
)

# Reforestation modules (new)
from src.reforestation.config import (
    MissionConfig, mangrove_preset, compute_sim_params, print_mission_summary
)
from src.reforestation.soil_detector import (
    detect_soil_mask, apply_tidal_mask,
    make_synthetic_tidal_grid, plot_soil_detection
)
from src.simulation.metrics import (
    compute_reforestation_metrics, print_reforestation_summary, plot_seed_drops
)
from src.viz.renderer import animate

os.makedirs('../results', exist_ok=True)
%matplotlib inline
print('Imports OK')

## 1. Configuration

All user-facing parameters live in `MissionConfig`.  
Operators change values here; everything else is derived automatically.

In [ ]:
# ── Real image path (set to None to use synthetic demo field) ──────────────
MANGROVE_IMAGE_PATH = '../aerial_mangrove_images/screenshot1.jpg'

# ── Field dimensions (real Abu Dhabi image: 500 m wide × 300 m tall) ──────
FIELD_W_M = 500.0     # metres wide
FIELD_H_M = 300.0     # metres tall
NCOLS     = 64
NROWS     = int(round(NCOLS * FIELD_H_M / FIELD_W_M))   # 38 — keeps cells nearly square

print(f'Grid: {NCOLS}×{NROWS}  ({FIELD_W_M/NCOLS:.1f} m × {FIELD_H_M/NROWS:.1f} m per cell)')

# ── User-configurable mission parameters ────────────────────────────────────
cfg = mangrove_preset()          # Distant Imagery / Abu Dhabi defaults

cfg.field_width_m         = FIELD_W_M
cfg.seed_spacing_m        = 1.5    # metres between seeds (0.5 dense ↔ 3.0 sparse)
cfg.battery_life_minutes  = 35.0   # 30–45 min typical
cfg.wind_speed_ms         = 0.0    # m/s — set >0 to simulate headwind drain penalty
cfg.n_drones              = 3
cfg.dock_positions        = [(0, 0)]

SPRAY_THRESH = 0.0               # soil mask already handles exclusions
ORIENTATION  = 0                 # strip direction (0=E-W; 90=N-S)
# ──────────────────────────────────────────────────────────────────────────

# Derive simulation params from config
params = compute_sim_params(cfg, ncols=NCOLS)
print_mission_summary(cfg, ncols=NCOLS)

# Build drone fleet
drones = [DroneSpec(id=i, seed_capacity=cfg.seed_capacity) for i in range(cfg.n_drones)]
print(f'Fleet: {cfg.n_drones} drones, {cfg.seed_capacity:,} seeds each')

## 2. Synthetic Mangrove Field

For the demo we generate a synthetic priority surface that approximates a mangrove
mudflat: irregular patches of exposed soil surrounded by water channels and canopy.

In production this comes from an aerial image via `detect_soil_mask()` (Section 3).

In [ ]:
def make_mangrove_synthetic(nrows=32, ncols=32, seed=7):
    """Synthetic field mimicking a mangrove mudflat.

    - Bright/tan patches = exposed soil (high priority to plant)
    - Near-zero / negative areas = water channels and tidal pools (excluded)
    - Moderate-high values = existing canopy (excluded via brightness threshold)
    """
    rng = np.random.default_rng(seed)
    base = np.zeros((nrows, ncols))

    # Soil patches (moderate-high values)
    for _ in range(6):
        cr, cc = rng.integers(4, nrows-4), rng.integers(4, ncols-4)
        sr, sc = rng.integers(3, 8),       rng.integers(3, 8)
        for r in range(nrows):
            for c in range(ncols):
                d = ((r - cr) / sr)**2 + ((c - cc) / sc)**2
                base[r, c] += rng.uniform(0.4, 0.75) * np.exp(-0.5 * d)

    # Water channels (low values — excluded by soil mask)
    for _ in range(3):
        orientation = rng.choice(['h', 'v'])
        pos   = rng.integers(4, nrows - 4)
        width = rng.integers(1, 3)
        if orientation == 'h':
            base[pos:pos+width, :] = rng.uniform(-0.3, 0.0, size=(width, ncols))
        else:
            base[:, pos:pos+width] = rng.uniform(-0.3, 0.0, size=(nrows, width))

    return np.clip(base, 0.0, 1.0)


# Synthetic field always built at NROWS×NCOLS so section 2 matches section 3
field_grid_synth = make_mangrove_synthetic(nrows=NROWS, ncols=NCOLS, seed=7)
soil_mask_synth  = field_grid_synth >= 0.2

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im0 = axes[0].imshow(field_grid_synth, cmap='YlOrBr', vmin=0, vmax=1, origin='upper')
axes[0].set_title('Synthetic field (priority surface)\nBright = exposed soil, Dark = water/canopy')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(soil_mask_synth.astype(int), cmap='RdYlGn', vmin=0, vmax=1, origin='upper')
axes[1].set_title(f'Soil mask (threshold=0.2)\n'
                  f'{soil_mask_synth.sum()} plantable / {NROWS*NCOLS} total cells')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

plt.suptitle(f'Synthetic Mangrove Field ({NCOLS}×{NROWS})', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Soil coverage: {soil_mask_synth.mean()*100:.1f}%  ({soil_mask_synth.sum()} plantable cells)')

## 3. Soil Mask — RGB/NDVI Detector

When a real aerial image is available, `detect_soil_mask()` classifies each pixel:

| Zone | Pseudo-NDVI | Blue channel | Classification |
|---|---|---|---|
| Open water / tidal channel | very low | **HIGH** | Non-plantable |
| Exposed mudflat / soil | **low** | low | **Plantable** ✓ |
| Existing canopy | **HIGH** | low | Non-plantable |

Two threshold cuts: `pseudo_ndvi < soil_ndvi_threshold` AND `blue < water_blue_threshold`

The mask plugs directly into `generate_strips(field_mask=soil_mask)` — the existing mechanism
already handles "fly over but don't seed" cells.

In [ ]:
from pathlib import Path
from src.reforestation.soil_detector import load_field_image

# Thresholds calibrated from pixel sampling of screenshot1.jpg:
#   Water (teal):   NDVI~0.22, Blue~0.56  → excluded by NDVI threshold 0.12
#   Mudflat (tan):  NDVI~0.00, Blue~0.44, Brightness~0.57  → plantable
#   Canopy (dark):  NDVI~-0.02, Blue~0.28, Brightness~0.39 → excluded by brightness
SOIL_NDVI_THRESH   = 0.12
WATER_BLUE_THRESH  = 0.50
MIN_BRIGHTNESS     = 0.42
MIN_PATCH_CELLS    = 2

use_real = MANGROVE_IMAGE_PATH and Path(MANGROVE_IMAGE_PATH).exists()

if use_real:
    print(f'Loading real image: {MANGROVE_IMAGE_PATH}')
    img_rgb = load_field_image(MANGROVE_IMAGE_PATH, nrows=NROWS, ncols=NCOLS)

    soil_mask, soil_priority, soil_meta = detect_soil_mask(
        MANGROVE_IMAGE_PATH,
        nrows                    = NROWS,
        ncols                    = NCOLS,
        soil_ndvi_threshold      = SOIL_NDVI_THRESH,
        water_blue_threshold     = WATER_BLUE_THRESH,
        min_brightness_threshold = MIN_BRIGHTNESS,
        min_patch_cells          = MIN_PATCH_CELLS,
    )
    field_grid = soil_priority

    print('Detection results:')
    for k, v in soil_meta.items():
        print(f'  {k}: {v:.3f}' if isinstance(v, float) else f'  {k}: {v}')

    # Four-panel diagnostic: RGB | NDVI | Brightness | mask
    import matplotlib.colors as mcolors
    R = img_rgb[:,:,0].astype(np.float32) / 255.0
    G = img_rgb[:,:,1].astype(np.float32) / 255.0
    B = img_rgb[:,:,2].astype(np.float32) / 255.0
    ndvi_arr   = (G - R) / (G + R + 1e-6)
    brightness = 0.299*R + 0.587*G + 0.114*B

    fig, axes = plt.subplots(1, 4, figsize=(22, 4))

    axes[0].imshow(img_rgb, aspect='auto')
    axes[0].set_title('Aerial Image (RGB)')
    axes[0].axis('off')

    im1 = axes[1].imshow(ndvi_arr, cmap='RdYlGn', vmin=-0.3, vmax=0.5, aspect='auto')
    axes[1].set_title(f'Pseudo-NDVI\n(threshold={SOIL_NDVI_THRESH})')
    axes[1].axis('off')
    plt.colorbar(im1, ax=axes[1], fraction=0.046)

    im2 = axes[2].imshow(brightness, cmap='YlOrBr', vmin=0, vmax=1, aspect='auto')
    axes[2].set_title(f'Brightness\n(min threshold={MIN_BRIGHTNESS})')
    axes[2].axis('off')
    plt.colorbar(im2, ax=axes[2], fraction=0.046)

    from matplotlib.patches import Patch
    cmap_m = mcolors.ListedColormap(['#a8d5e2', '#c8a97a'])
    axes[3].imshow(soil_mask.astype(int), cmap=cmap_m, vmin=0, vmax=1, aspect='auto')
    axes[3].set_title(
        f'Plantable Soil Mask\n'
        f'soil={soil_meta["soil_pct"]:.1%}  water={soil_meta["water_pct"]:.1%}  '
        f'dark-canopy={soil_meta["dark_canopy_pct"]:.1%}'
    )
    axes[3].axis('off')
    axes[3].legend(handles=[
        Patch(facecolor='#c8a97a', label=f'Plantable ({soil_meta["plantable_cells"]} cells)'),
        Patch(facecolor='#a8d5e2', label='Skip (water / canopy)'),
    ], loc='lower right', fontsize=7)

    plt.suptitle(f'Soil Detection — Abu Dhabi Mangrove ({NCOLS}×{NROWS} grid)', fontsize=12)
    plt.tight_layout()
    plt.show()

else:
    # Fallback: synthetic demo
    print('Real image not found — using synthetic soil mask.')
    img_rgb    = None
    soil_mask  = soil_mask_synth
    field_grid = field_grid_synth * soil_mask_synth.astype(float)
    soil_meta  = {'plantable_cells': int(soil_mask.sum())}

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.imshow(soil_mask.astype(int), cmap='RdYlGn', vmin=0, vmax=1, origin='upper')
    ax.set_title(f'Synthetic soil mask — {soil_mask.sum()} plantable cells')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

print(f'\nField grid shape : {np.array(field_grid).shape}')
print(f'Soil mask shape  : {soil_mask.shape}')
print(f'Plantable cells  : {soil_mask.sum()} / {NROWS*NCOLS} ({soil_mask.mean()*100:.1f}%)')

## 4. Strip Generation

Boustrophedon strips with `field_mask=soil_mask`.  
Drones fly every strip but only dispense seeds on soil cells.

## 4b. Contour Planting — Sinuous Tidal-Channel Paths

Expert practitioners avoid straight lawnmower rows because uniform grids:
- Create **artificial drainage channels** that cause erosion
- Produce **uniform canopy density** — lower biodiversity than patchy natural growth
- Miss **hydrological sweet spots** (slightly elevated mounds where survival is higher)

Instead, drones follow the **natural contours of the tidal zone** — planting outward from the water edge inward, mirroring how mangroves actually colonise mudflats.

**Implementation:** `generate_contour_strips()` runs a distance transform on the soil mask (`scipy.ndimage.distance_transform_edt`). Each cell's value = its distance (in cells) to the nearest water/canopy boundary. Cells are bucketed into concentric rings; within each ring, cells are ordered by angle around the ring centroid — tracing the perimeter of the ring rather than scanning it row by row.

| Parameter | Effect |
|---|---|
| `strip_width=1` | Maximum sinuosity — many narrow rings, slower MILP |
| `strip_width=2` | Recommended — smooth curves, fewer strips |
| `strip_width=3` | Wider swaths, less curve detail, fastest MILP |

In [ ]:
from src.field.contour import generate_contour_strips

# Generate contour strips from the same soil mask
# field_grid is the priority surface (numpy array in both real and synthetic paths)
contour_strips = generate_contour_strips(
    soil_mask,
    np.array(field_grid),        # priority surface — same grid used by boustrophedon
    strip_width      = 1,        # width=1 → finest rings, most sinuous
    seconds_per_cell = cfg.seconds_per_cell,
)

print(f'Boustrophedon strips : {len(strips):>4}')
print(f'Contour strips (w=1) : {len(contour_strips):>4}')
print()
if contour_strips:
    print('Contour strip summary:')
    for s in contour_strips[:5]:
        print(f'  strip {s.id:>2}: {len(s.cells):>4} cells, priority={s.priority:.3f}')
    if len(contour_strips) > 5:
        print(f'  ... ({len(contour_strips) - 5} more)')

# ── Side-by-side path visualisation ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cmap_strips = plt.get_cmap('tab20')

for ax, strip_list, title in [
    (axes[0], strips,         'Boustrophedon (lawnmower)'),
    (axes[1], contour_strips, 'Contour (tidal-channel-following)'),
]:
    # Background: soil mask (green = plantable, dark = water/canopy)
    bg = np.where(soil_mask, 0.85, 0.25)
    ax.imshow(bg, cmap='Greens', vmin=0, vmax=1, origin='upper', alpha=0.6)

    # Draw each strip's traversal path
    for s in strip_list:
        if not s.cells:
            continue
        rows = [r for r, c in s.cells]
        cols = [c for r, c in s.cells]
        colour = cmap_strips(s.id % 20)
        ax.plot(cols, rows, '-', color=colour, linewidth=1.4, alpha=0.85)
        # Spray cells as dots
        spray_set = set(s.spray_cells)
        sr = [r for r, c in s.cells if (r, c) in spray_set]
        sc = [c for r, c in s.cells if (r, c) in spray_set]
        ax.scatter(sc, sr, s=5, color=colour, alpha=0.5, linewidths=0)

    ax.set_title(
        f'{title}\n'
        f'{len(strip_list)} strips · {sum(len(s.spray_cells) for s in strip_list)} spray cells',
        fontsize=11,
    )
    ax.set_xlim(-0.5, ncols - 0.5)
    ax.set_ylim(nrows - 0.5, -0.5)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_aspect('equal')

plt.suptitle(
    'Strip Path Comparison — same soil mask, different traversal strategy\n'
    'Each colour = one strip assignment',
    fontsize=12, fontweight='bold',
)
plt.tight_layout()
plt.show()

In [ ]:
from src.reforestation.soil_detector import apply_tidal_mask

# Convenience aliases (downstream cells use lowercase nrows / ncols)
nrows, ncols = NROWS, NCOLS

strips = generate_strips(
    field_grid,
    seconds_per_cell = cfg.seconds_per_cell,
    orientation_deg  = ORIENTATION,
    spray_threshold  = SPRAY_THRESH,
    field_mask       = soil_mask,
)

total_spray_cells = sum(len(s.spray_cells) for s in strips)
print(f'Strips generated   : {len(strips)}')
print(f'Soil cells to seed : {total_spray_cells}')
print(f'Transit-only cells : {sum(len(s.cells)-len(s.spray_cells) for s in strips)}')

# Overlay strip paths on soil mask / aerial image
fig, ax = plt.subplots(figsize=(10, 5))
if img_rgb is not None:
    ax.imshow(img_rgb, aspect='auto', alpha=0.6, origin='upper',
              extent=(-0.5, ncols-0.5, nrows-0.5, -0.5))
else:
    ax.imshow(soil_mask.astype(int), cmap='Greens', vmin=0, vmax=1.5,
              origin='upper', alpha=0.5)

palette = plt.cm.tab20(np.linspace(0, 1, max(len(strips), 1)))
for i, s in enumerate(strips):
    if s.cells:
        ax.plot([c for _, c in s.cells], [r for r, _ in s.cells],
                color=palette[i % len(palette)], linewidth=0.7, alpha=0.8)
    for r, c in s.spray_cells:
        ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1,
                     facecolor=palette[i % len(palette)], alpha=0.30,
                     edgecolor='none', zorder=2))

ax.set_xlim(-0.5, ncols-0.5)
ax.set_ylim(nrows-0.5, -0.5)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f'Strip paths on soil mask — {len(strips)} strips, '
             f'{total_spray_cells} plantable cells  '
             f'(coloured = seed-active, line-only = transit)')
plt.tight_layout()
plt.show()

## 5. Fleet Assignment (MILP)

Unchanged from the spraying system.  Minimize makespan across 3 drones.  
Seed capacity is a simulation-layer concern, not an optimizer constraint.

In [ ]:
result = assign_strips(strips, drones, objective_mode='makespan')

print(f'Optimizer status : {result.status}')
print(f'Solve time       : {result.solve_time:.2f}s')
print(f'Planned makespan : {result.makespan:.0f}s')
print()
for d_id, strip_ids in result.assignment.items():
    total_time = sum(strips_by_id[sid].time
                     for sid in strip_ids
                     if (strips_by_id := {s.id: s for s in strips}))
    print(f'  Drone {d_id}: {len(strip_ids)} strips  ({total_time:.0f}s workload)')

strips_by_id = {s.id: s for s in strips}
for d_id, strip_ids in result.assignment.items():
    total_time = sum(strips_by_id[sid].time for sid in strip_ids)
    n_seeds_est = sum(len(strips_by_id[sid].spray_cells) for sid in strip_ids) * params['seeds_per_cell']
    print(f'  Drone {d_id}: {len(strip_ids):2d} strips  '
          f'{total_time:6.0f}s  ~{n_seeds_est:.0f} seeds (before refills)')

## 6. Simulation with Seed Tracking

Two new parameters activate reforestation mode:
- `seeds_per_cell` — seeds dispensed per spray cell (derived from spacing + field scale)
- `seed_jitter_sigma` — Gaussian offset for each drop position (in grid cells)

Drones return to dock when the seed hopper hits 0, refill, then resume — same lifecycle as battery.

In [ ]:
print('Simulation parameters:')
for k, v in params.items():
    print(f'  {k:<28}: {v}')
print()

state_history = simulate(
    strips                = strips,
    drones                = drones,
    result                = result,
    nrows                 = nrows,
    ncols                 = ncols,
    battery_drain_per_cell= params['battery_drain_per_cell'],
    recharge_time_steps   = params['recharge_time_steps'],
    dock_positions        = cfg.dock_positions,
    seeds_per_cell        = params['seeds_per_cell'],
    seed_jitter_sigma     = params['seed_jitter_sigma'],
)

print(f'Simulation complete: {len(state_history)} timesteps')

# Count seed-related events
hopper_events = [s for s in state_history
                 if s.get('event') and 'seed hopper empty' in s['event']]
battery_events = [s for s in state_history
                  if s.get('event') and 'battery' in (s.get('event') or '').lower()]
print(f'Hopper-empty returns : {len(hopper_events)}')
print(f'Battery-related stops: {len(battery_events)}')

## 7. Reforestation Metrics

Operator-facing summary with seed-specific KPIs.

In [ ]:
metrics = compute_reforestation_metrics(
    state_history,
    strips,
    nrows, ncols,
    meters_per_cell = params['meters_per_cell'],
    survival_rate   = cfg.survival_rate,
    target_density_per_m2 = cfg.target_density_per_m2,
)

print_reforestation_summary(metrics, config=cfg)

# Seed series plot (seed hopper level per drone over time)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for d_id, series in metrics.get('seed_series', {}).items():
    axes[0].plot(series, label=f'Drone {d_id}')
axes[0].set_title('Seed hopper level over time')
axes[0].set_xlabel('Timestep')
axes[0].set_ylabel('Seeds remaining')
axes[0].axhline(0, color='red', linestyle='--', alpha=0.4, linewidth=1)
axes[0].legend()
axes[0].grid(alpha=0.3)

for d_id, series in metrics.get('battery_series', {}).items():
    axes[1].plot(series, label=f'Drone {d_id}')
axes[1].set_title('Battery level over time')
axes[1].set_xlabel('Timestep')
axes[1].set_ylabel('Battery (%)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Resource Depletion: Seeds + Battery', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Animation

Renders the mission with:
- `mode_label`: shows "Mangrove Reforestation" in title
- `show_seed_drops=True`: static purple cloud of all seed drop positions
- Info panel: battery % and seed count per drone

In [ ]:
GIF_PATH = '../results/phase5_mangrove_reforestation.gif'

# Subsample to keep GIF small (~120 frames)
FRAME_SKIP  = max(1, len(state_history) // 120)
gif_history = state_history[::FRAME_SKIP]

n_plantable = sum(len(s.spray_cells) for s in strips)   # viable planting cells only

anim = animate(
    state_history     = gif_history,
    nrows             = nrows,
    ncols             = ncols,
    interval_ms       = 150,
    dock_positions    = cfg.dock_positions,
    background_image  = img_rgb,
    save_path         = GIF_PATH,
    show              = False,
    show_seed_drops   = True,
    mode_label        = cfg.name,
    n_plantable_cells = n_plantable,
)

print(f'GIF saved : {GIF_PATH}')
print(f'Frames    : {len(gif_history)}  (every {FRAME_SKIP} of {len(state_history)} steps)')
print(f'Plantable cells used as coverage denominator: {n_plantable}')

# Inline preview: start vs end frame
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, s, label in [(axes[0], state_history[0], 'Start'),
                      (axes[1], state_history[-1], 'End')]:
    grid_arr = np.array(s['grid'])
    if img_rgb is not None:
        ax.imshow(img_rgb, aspect='auto', alpha=0.55, origin='upper',
                  extent=(-0.5, ncols-0.5, nrows-0.5, -0.5))
    ax.imshow(grid_arr, cmap='RdYlGn_r', vmin=0, vmax=3,
              origin='upper', alpha=0.6 if img_rgb is not None else 1.0)
    ax.set_title(f'{label} — t={s["timestep"]}')
    ax.axis('off')
plt.suptitle('Mission Snapshot: Start vs End', fontsize=12)
plt.tight_layout()
plt.show()

## 9. Seed Drop Scatter

All recorded seed positions across the mission — with Gaussian jitter applied.  
Shows the **natural, non-grid-aligned distribution** that mimics organic seed dispersal.

In [ ]:
total_drops = sum(len(s.get('seed_drops', [])) for s in state_history)
print(f'Total seed drop events recorded: {total_drops:,}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: seed cloud on aerial photo (or plain grid background)
ax_left = axes[0]
if img_rgb is not None:
    ax_left.imshow(img_rgb, aspect='auto', alpha=0.75, origin='upper',
                   extent=(-0.5, ncols-0.5, nrows-0.5, -0.5))

all_xs, all_ys = [], []
for s in state_history:
    for drop in s.get('seed_drops', []):
        ar, ac = drop['actual']
        all_xs.append(ac)
        all_ys.append(ar)

if all_xs:
    ax_left.scatter(all_xs, all_ys, s=1.5, alpha=0.07, color='#4a148c', linewidths=0)

ax_left.set_xlim(-0.5, ncols-0.5)
ax_left.set_ylim(nrows-0.5, -0.5)
ax_left.set_title(f'Seed cloud on {"aerial image" if img_rgb is not None else "grid"}\n'
                  f'({len(all_xs):,} drops, Gaussian jitter σ={cfg.seed_jitter_sigma} cells)')
ax_left.axis('off')

# Right: density heatmap
if total_drops > 0:
    density = np.zeros((nrows, ncols))
    for s in state_history:
        for drop in s.get('seed_drops', []):
            r, c = drop['actual']
            density[r, c] += 1

    im = axes[1].imshow(density, cmap='YlOrRd', origin='upper', aspect='auto')
    axes[1].set_title(f'Seed density heatmap\n(max {density.max():.0f} seeds / cell)')
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1], fraction=0.046, label='Seeds / cell')

plt.suptitle(
    f'Non-Linear Planting — {total_drops:,} drops  |  '
    f'{metrics.get("total_seeds_planted", 0):,} seeds planted  |  '
    f'{metrics.get("total_area_covered_m2", 0):,.0f} m² covered',
    fontsize=11,
)
plt.tight_layout()
plt.show()

# Verify jitter is working
nominal_actual_match = sum(
    1 for s in state_history
    for d in s.get('seed_drops', [])
    if d['nominal'] == d['actual']
)
print(f'Drops at exact cell centre: {nominal_actual_match}/{total_drops} '
      f'({100*nominal_actual_match/max(total_drops,1):.1f}%)')
print('(Should be < 100% when jitter is active)')

## 10. Tidal Event — Dynamic Mask + Replanning

A rain event or tide change floods the bottom quarter of the field.  
We call `apply_tidal_mask()` to update the plantable area, regenerate strips,
and trigger a weather-replan event at timestep 30.

In [ ]:
# Build tidal grid: bottom 25% of field floods
tidal_grid = make_synthetic_tidal_grid(
    nrows, ncols,
    flooded_rows=(int(nrows * 0.75), nrows),
    flood_value=0.8,
    seed=42
)

# Updated plantable mask
tidal_soil_mask = apply_tidal_mask(soil_mask, tidal_grid,
                                    tidal_threshold=cfg.tidal_threshold)

# Visualise the change
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(tidal_grid, cmap='Blues', vmin=0, vmax=1, origin='upper')
axes[0].set_title('Tidal grid (0=dry, 1=flooded)')
axes[0].axis('off')

axes[1].imshow(soil_mask.astype(int), cmap='Greens', vmin=0, vmax=1, origin='upper')
axes[1].set_title(f'Original soil mask\n{soil_mask.sum()} plantable cells')
axes[1].axis('off')

axes[2].imshow(tidal_soil_mask.astype(int), cmap='Greens', vmin=0, vmax=1, origin='upper')
axes[2].set_title(f'Post-tidal mask\n{tidal_soil_mask.sum()} plantable cells'
                   f' (-{soil_mask.sum()-tidal_soil_mask.sum()} flooded)')
axes[2].axis('off')

plt.suptitle('Tidal Event: Flooded cells excluded from planting', fontsize=12)
plt.tight_layout()
plt.show()

# Re-generate strips with tidal mask and run simulation with weather-replan event
tidal_strips = generate_strips(
    field_grid,
    seconds_per_cell = cfg.seconds_per_cell,
    orientation_deg  = ORIENTATION,
    spray_threshold  = 0.0,
    field_mask       = tidal_soil_mask,
)
tidal_result = assign_strips(tidal_strips, drones, objective_mode='makespan')

# Weather replan event at timestep 30 using the tidal mask strips
weather_event = [{
    'timestep': 30,
    'type': 'weather_replan',
}]

tidal_history = simulate(
    strips                = tidal_strips,
    drones                = drones,
    result                = tidal_result,
    nrows                 = nrows,
    ncols                 = ncols,
    failure_events        = weather_event,
    battery_drain_per_cell= params['battery_drain_per_cell'],
    recharge_time_steps   = params['recharge_time_steps'],
    dock_positions        = cfg.dock_positions,
    seeds_per_cell        = params['seeds_per_cell'],
    seed_jitter_sigma     = params['seed_jitter_sigma'],
)

tidal_metrics = compute_reforestation_metrics(
    tidal_history, tidal_strips, nrows, ncols,
    meters_per_cell=params['meters_per_cell'],
)

print('\nTidal mission results:')
print(f'  Plantable cells    : {tidal_soil_mask.sum()} (vs {soil_mask.sum()} baseline)')
print(f'  Coverage           : {tidal_metrics["coverage_pct"]}%')
print(f'  Seeds planted      : {tidal_metrics.get("total_seeds_planted", "–")}  '
      f'(vs {metrics.get("total_seeds_planted", "–")} baseline)')
print(f'  Makespan           : {tidal_metrics["makespan"]} steps')

## 11. Failure + Replanning

Drone 1 suffers a mechanical failure at timestep 20.  
The MILP replanner redistributes remaining strips to the two surviving drones.

In [ ]:
failure_events = [{'timestep': 20, 'drone_id': 1, 'type': 'mechanical'}]

failure_history = simulate(
    strips                = strips,
    drones                = drones,
    result                = result,
    nrows                 = nrows,
    ncols                 = ncols,
    failure_events        = failure_events,
    battery_drain_per_cell= params['battery_drain_per_cell'],
    recharge_time_steps   = params['recharge_time_steps'],
    dock_positions        = cfg.dock_positions,
    seeds_per_cell        = params['seeds_per_cell'],
    seed_jitter_sigma     = params['seed_jitter_sigma'],
)

fail_metrics = compute_reforestation_metrics(
    failure_history, strips, nrows, ncols,
    meters_per_cell=params['meters_per_cell'],
)

print('Baseline vs failure comparison:')
comparison_keys = [
    ('coverage_pct', '%'),
    ('total_seeds_planted', 'seeds'),
    ('total_area_covered_m2', 'm²'),
    ('makespan', 'steps'),
    ('time_to_recovery', 'steps'),
]
print(f"  {'Metric':<28} {'Baseline':>12} {'With Failure':>14}")
print(f"  {'-'*56}")
for key, unit in comparison_keys:
    b = metrics.get(key, '–')
    f = fail_metrics.get(key, '–')
    print(f"  {key:<28} {str(b):>12} {str(f):>14}  {unit}")

# Coverage over time comparison
fig, ax = plt.subplots(figsize=(10, 4))
total_spray = sum(len(s.spray_cells) for s in strips)
for hist, label, color in [
    (state_history,   'Baseline (no failure)', '#1565c0'),
    (failure_history, 'Drone 1 failure @ t=20', '#e65100'),
]:
    pct = [100 * sum(1 for row in s['grid'] for c in row if c==2) / total_spray
           for s in hist]
    ax.plot(pct, color=color, label=label)

ax.axvline(20, color='red', linestyle='--', alpha=0.5, label='Failure event')
ax.set_xlabel('Timestep')
ax.set_ylabel('Seed coverage (%)')
ax.set_title('Coverage over time: baseline vs. drone failure + replanning')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Monte Carlo Worst-Case

50 runs with random failures and varying battery drain.  
**Primary operator question:** How bad can the mission get?

In [ ]:
print('Running Monte Carlo analysis (50 runs)...')

mc = monte_carlo_analysis(
    strips                 = strips,
    drones                 = drones,
    result                 = result,
    nrows                  = nrows,
    ncols                  = ncols,
    n_runs                 = 50,
    failure_prob_per_drone = 0.20,
    battery_drain_range    = (params['battery_drain_per_cell'] * 0.8,
                              params['battery_drain_per_cell'] * 1.5),
    seed                   = 42,
    seeds_per_cell         = params['seeds_per_cell'],
    seed_jitter_sigma      = params['seed_jitter_sigma'],
)

print(f"\nResults across {mc['n_runs']} runs:")
print(f"  Completion rate      : {mc['completion_rate']}%")
print(f"  Coverage mean / P5   : {mc['coverage_pct_mean']}% / {mc['coverage_pct_p5']}%")
print(f"  Makespan mean / P95  : {mc['makespan_mean']} / {mc['makespan_p95']} steps")
print(f"  Time-to-recovery P95 : {mc.get('time_to_recovery_p95', 'N/A')} steps")
print(f"  Fleet-failed runs    : {mc['fleet_failed_runs']}")

plot_monte_carlo(mc, title='Monte Carlo — Mangrove Reforestation')
plt.tight_layout()
plt.show()

## 13. Summary

| What changed | How |
|---|---|
| **Seed capacity** | `DroneSpec.seed_capacity`; depletes during spraying; refills at dock alongside battery |
| **Dispersal rate** | `seeds_per_cell = cell_area_m² / seed_spacing_m²`; user sets spacing in metres |
| **Soil-only planting** | `detect_soil_mask()` → `field_mask` → `generate_strips()` (existing mechanism) |
| **Non-straight planting** | Gaussian jitter on recorded `seed_drops` positions (navigation unchanged) |
| **Contour strips** | `generate_contour_strips()` — EDT distance rings hug tidal channel edges; same `Strip` format, works with existing MILP and sim |
| **Battery model** | `battery_life_minutes` → `battery_drain_per_cell` via `compute_sim_params()` |
| **Tidal events** | `apply_tidal_mask()` + regenerate strips + weather-replan event |
| **Wind events (MCP)** | `report_wind_change()` updates `battery_drain_per_cell` and `seed_jitter_sigma` for next sim chunk |
| **New metrics** | `total_seeds_planted`, `area_covered_m²`, `expected_survivors`, `seed_series` |
| **GIF metrics** | Title shows `X% soil seeded | N seeds planted`; info panel shows per-drone seed load |

| What's unchanged | Reason |
|---|---|
| MILP optimizer | Seed capacity isn't an assignment constraint |
| Three-tier planner (FULL/DEGRADED/HEURISTIC) | Failure modes are identical |
| Failure injection & replanning | Hopper-empty uses the same `_initiate_return()` path as battery |
| Overlay visualization | Cell states (0/1/2/3) map naturally to unplanted/active/seeded/failed |
| All existing notebooks | New params default to 0 (spray mode) — fully backward compatible |

**Seed dispersal model note:** real Distant Imagery drones drop seeds under gravity (no cannon). Jitter models forward drift during the fall (`dx ≈ v·√(2h/g) ≈ 5 m at h=5 m, v=5 m/s`) plus wind scatter. `seed_jitter_sigma=0.3` cells is a conservative lower bound — see `config.py` and `engine.py` comments for the full derivation.

---
**Next steps (Phase 6)**
- Real aerial mangrove imagery (NAIP, OpenAerialMap, Copernicus)
- Reseeding simulation across 2–3 visits with 40% survival
- Multi-dock layout for large coastal sites
- Streamlit dashboard